In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'operational' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
gapCases = pl.read_parquet(dataPrepPath / 'opticalGapCases.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
baseRows = predictions.join(gapCases.select('rowId', 'ndvi', 'locfNdvi', 'interpolationNdvi', 'gapAgeDays', 'bracketingGapDays', 'locfEvaluable', 'interpolationEvaluable'), on='rowId', how='inner').with_columns((pl.col('prediction') - pl.col('ndvi')).abs().alias('radarAbsError'))
locfRows = baseRows.filter(pl.col('locfEvaluable')).select('model', 'featureSet', 'seed', pl.lit('locf').alias('baseline'), pl.col('gapAgeDays').cast(pl.Float64).alias('gapLengthDays'), (pl.col('radarAbsError') - (pl.col('locfNdvi') - pl.col('ndvi')).abs()).alias('errorDifference'))
interpolationRows = baseRows.filter(pl.col('interpolationEvaluable')).select('model', 'featureSet', 'seed', pl.lit('interpolation').alias('baseline'), pl.col('bracketingGapDays').cast(pl.Float64).alias('gapLengthDays'), (pl.col('radarAbsError') - (pl.col('interpolationNdvi') - pl.col('ndvi')).abs()).alias('errorDifference'))
crossoverRows = pl.concat([locfRows, interpolationRows])
crossoverEstimate = crossoverRows.group_by('model', 'featureSet', 'seed', 'baseline').agg(pl.len().alias('rowCount'), pl.col('gapLengthDays').min().alias('minimumGapDays'), pl.col('gapLengthDays').max().alias('maximumGapDays'), pl.col('gapLengthDays').mean().alias('meanGapDays'), pl.col('errorDifference').mean().alias('meanErrorDifference'), pl.col('gapLengthDays').var(ddof=0).alias('gapVariance'), pl.cov('gapLengthDays', 'errorDifference', ddof=0).alias('gapErrorCovariance')).with_columns((pl.col('gapErrorCovariance') / pl.col('gapVariance')).alias('slope')).with_columns((pl.col('meanErrorDifference') - pl.col('slope') * pl.col('meanGapDays')).alias('intercept')).with_columns(pl.when(pl.col('slope') < 0).then(-pl.col('intercept') / pl.col('slope')).otherwise(None).alias('crossoverDays')).with_columns(pl.col('crossoverDays').is_between(pl.col('minimumGapDays'), pl.col('maximumGapDays'), closed='both').alias('crossoverInObservedRange')).sort('baseline', 'model', 'featureSet', 'seed')
crossoverEstimate.write_parquet(outputPath / 'gapLengthCrossover.parquet')
assert crossoverEstimate.height == 2 * 2 * 4 * 5
assert crossoverEstimate.filter(pl.col('rowCount') <= 0).is_empty()
crossoverEstimate
